# Synchronization

## I. Sync of streams (diff kernels)

### 1. Pageable vs Pinned

Swaped from RAM to Disk vs Stays in RAM

```
integer :: n, istat
logical :: pinnedFlag
real, allocatable, pinned :: a_h(:)
...
allocate(a_h(n), STAT = istat, PINNED = pinnedFlag)
if (istat /= 0) print *, 'Allocation of a_h failed'
if (.not. pinnedFlag) &
  print *, 'Pinned allocation of a_h failed```

nsys starts -report cuda_gpu_trace

### 2. Streams

Parallel but syncronized with the null stream. Also need to sync data transfers

In [ ]:
!nvfortran sync_str/1_streams.cuf
!./a.out

- Parrallel data transfers (dest, source, size, stream)

```
istat = cudaMemcpyAsync(a_d, a, n, 0)
call kernel<<<gridSize,blockSize>>>(a_d)
call cpuRoutine(b)

In [ ]:
!nvfortran sync_str/2_streams.cuf
!./a.out

##### a. cudaStreamSynchronize

- cudaDeviceSynchronize: Sync all device ops
- cudaStreamSyncronize: Sync one stream

In [ ]:
!nvfortran sync_str/3_sync_str.cuf
!./a.out

##### b. cudaEventSynchronize 

- Syncronizes streams under an event
- Event record -> records the end, thats what syncronizes

-> eventrecord -> sync -> launch : putting blocks
- need to record event to sync
- eventsync wait for event to be recorded

In [ ]:
!nvfortran sync_str/4_sync_ev.cuf
!./a.out

##### c. Default Stream

- Set default stream for data transfers and kernel executionsv

In [ ]:
!nvfortran sync_str/5_def_str.cuf
!./a.out

You can declare for which variable is the default stream

- Only work is variable is allocatable
- cudaforSetDefaultStream(a_d, s1) : only for allocation, not overrided
- minval(a_d, stream=s1)

In [ ]:
!nvfortran sync_str/6_def_str_al.cuf
!./a.out

#### d. Non blocking stream

They don't sync with the default stream, non-blocking

- cudaStreamCreateWithFlags(stream, cudaStreamNonBlocking)

- cudaStreamCreateWithFlags(stream, cudaStreamDefault) -> to sync with null stream

In [ ]:
!nvfortran sync_str/7_nb_str.cuf
!./a.out

- In this example with cudaStreamDefault, streams 4-6 are blocked by stream 3 (null = 0)

In [ ]:
!nvfortran sync_str/8_null_default.cuf
!./a.out

## II. Sync of threads (within kernel)

Through shared, mult have shared mem

- With the shared param

call kernel<<<grid, blocks, sharedMemBytes>>>

- grid size: (x,y,z)
- block size: (x,y,z)
- mem: shared size

Recommended: declare whole mem, but each array size separetaly

In [ ]:
!nvfortran sync_thr/1_shared.cuf
!./a.out

#### a. Sync Threads in a Block

- syncthreads()

- syncthreads_and() -> true if all true

- syncthreads_or() -> ture if only one true

- syncthread_count () -> count of which are true

#### b. Sync Warps

- warpID = (threadIdx%x-1)/warpsize+1
- laneID = threadIdx%x-1-(warpID-1)*warpsize

- syncwarp(mask)

Create and sync mask:

- ballot(predicate)
- ballot_sync(mask, predicate)

Need to create mask and then create it -> syncs all threads in warps that inside mask

- syncwarp(): all in warp

In [ ]:
!nvfortran sync_thr/2_shared_wp.cuf
!./a.out

#### c. SHLF funcs

Comm info between threads, bring the `var` variable from other thread in warp

- __shfl(var, srcLane, ...): bring the var variable from the srcLane in the warp

- __shfl_xor(var, laneMask) -> creates the srcLane from xor (laneID-1, laneMask) + 1



In [ ]:
!nvfortran sync_thr/3_shfl.cuf
!./a.out

##### d. Atomic ops

To avoid race condtions

- atomicAdd(mem, val)

- atomicSub(mem, val)

- atomicMax(mem, val1, val2)


In [ ]:
!nvfortran sync_thr/4_atomic.cuf
!./a.out

#### e. Mem fences

Fence memory writes before any other thread can:

- threadfence_block(): within block
- threadfence(): within all threads

stops all exec until its mem ops finish

In [ ]:
!nvfortran sync_thr/5_thr_bl.cuf
!./a.out

#### f. Coop groups

Diff groups, each with rank and size. Helps syncthreads accross this gropus

- grid_groups
- cluster_group
- thread_group
- coalesced_group -> warp

```use cooperative_groups

type(thread_block) :: tb
type(grid_group)   :: gg
type(thread_block_tile) :: warp
type(cluster_group) :: cluster

tb   = this_block()   ! group of all threads in a block
gg   = this_grid()    ! group of all threads in the grid
warp = this_warp()    ! group of all threads in a warp
clus = this_cluster() ! group of all threads in a cluster

call syncthreads(tb)   ! block-wide sync
call syncthreads(gg)   ! grid-wide sync
call syncthreads(warp) ! warp-wide sync
```


Syncs threads in a wider kind of group

##### f.1 Grid sync

In [ ]:
!nvfortran sync_thr/6_grid_sync.cuf
!./a.out

##### f.2 Thr block clusters

- Number of thread blocks that comm -> share mem
- ```cluster_group```
- cluster_map_shared_rank(var, blockIdx.x, blockIdx.y, blockIdx.z)
- pointer(pointer, pointee_array) -> ```ds```

In [ ]:
!nvfortran sync_thr/7_clus.cuf
!./a.out